# C5-00a · IRIS — ETL, Feature Engineering y Preprocessing
### Capítulo 5 · Machine Learning 1: Aprendizaje supervisado · Samsung Innovation Campus
**Puente Probabilidad & Estadística → Machine Learning**

**Tipo de caso:** clasificación **multiclase**, limpia y balanceada: el *caso de laboratorio*.
Pocas dimensiones, sin nulos, sin categóricas: ideal para ver cada paso sin ruido.

**Objetivos**
1. Hacer un ETL mínimo y reproducible.
2. Leer el dataset como una muestra de una distribución conjunta P(X, Y).
3. Crear features justificadas por un hallazgo estadístico.
4. Armar un preprocesador que se ajusta **solo en train**.
5. Detectar **leakage** cuando cambia la tarea (clasificación → regresión).
6. Exportar el módulo `cap5_iris.py`.

**Cómo encaja este Colab en el capítulo**

```
CAPA BASE (este Colab)                      CAPA MODELO
ETL -> Feature Engineering -> Preprocessing -> módulo cap5_iris.py -> C5-M01 ... C5-M12
```

Al final se exporta el módulo **`cap5_iris.py`** con la **misma interfaz** que los otros dos datasets:

| Función | Devuelve |
|---|---|
| `cargar(tarea)` | `X` (DataFrame) e `y` (Series) listos para modelar |
| `preprocesador(tarea, ...)` | un `Pipeline` / `ColumnTransformer` **sin ajustar** |
| `dividir(X, y, tarea)` | train/test (estratificado si es clasificación) |
| `INFO` | ficha del dataset: tipo de caso, tareas, métrica sugerida |

Los Colabs de modelos solo cambian `DATASET = "iris"` y reutilizan todo esto.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

SEED = 42
pd.set_option("display.precision", 3)

## 2. ETL

| Etapa | Qué hacemos | Puente P&E |
|---|---|---|
| **E**xtract | `load_iris(as_frame=True)` | obtenemos una **muestra** de n = 150 |
| **T**ransform | nombres cortos, tipos, duplicados | definimos las **variables aleatorias** |
| **L**oad | DataFrame limpio en memoria / CSV | la muestra queda fija y reproducible |

In [ ]:
# Extract
crudo = load_iris(as_frame=True)
df = crudo.frame.copy()

# Transform
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "especie"]
CLASES = list(crudo.target_names)

print("Forma:", df.shape)
print("Clases:", dict(enumerate(CLASES)))
df.head()

In [ ]:
# Ficha de calidad
ficha = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "valores_unicos": df.nunique(),
})
print("Filas duplicadas:", df.duplicated().sum())
ficha

> **Puente P&E — ¿duplicado = error?**
> Iris tiene **1 fila duplicada**. Con mediciones redondeadas a 0.1 cm, dos flores distintas
> pueden coincidir. **No se elimina**: borrarla cambiaría la muestra sin evidencia de error de carga.

In [ ]:
# Load: guardamos la versión limpia (útil para auditar)
df.to_csv("iris_limpio.csv", index=False)

## 3. EDA orientado a decisiones

No es un EDA exhaustivo: buscamos **hallazgos que justifiquen** decisiones de FE y preprocessing.

In [ ]:
# 3.1 Balance de clases -> prior P(Y = k)
balance = df["especie"].value_counts(normalize=True).sort_index().rename("P(Y=k)")
balance.index = CLASES
balance.to_frame()

> **Puente P&E:** clases 50/50/50 ⇒ **prior uniforme** P(Y = k) = 1/3.
> El *baseline* que predice siempre la clase más frecuente acierta 33%: cualquier modelo tiene que superar eso.

In [ ]:
# 3.2 Momentos de la distribución condicional X | Y = k
numericas = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
por_clase = df.groupby("especie")[numericas].agg(["mean", "std"])
por_clase.index = CLASES
por_clase

In [ ]:
# 3.3 ¿Qué feature separa mejor las especies? ANOVA de un factor
filas = []
for c in numericas:
    grupos = [g[c].to_numpy() for _, g in df.groupby("especie")]
    F, p = stats.f_oneway(*grupos)
    filas.append({"feature": c, "F": F, "p_valor": p,
                  "asimetria": df[c].skew(), "cv_%": 100 * df[c].std() / df[c].mean()})
anova = pd.DataFrame(filas).sort_values("F", ascending=False).set_index("feature")
anova

> **Puente P&E:** una F alta indica que la **variabilidad entre especies** supera ampliamente
> la variabilidad dentro de cada especie. Las features de pétalo dominan.
> La asimetría de todas está lejos de 1 ⇒ **no hace falta transformar con log**.

In [ ]:
# 3.4 Correlación global vs. dentro de cada especie (paradoja de Simpson)
global_ = df[numericas].corr().loc["sepal_length", "sepal_width"]
dentro = df.groupby("especie")[["sepal_length", "sepal_width"]].corr().xs("sepal_length", level=1)["sepal_width"]
dentro.index = CLASES
pd.concat([pd.Series({"GLOBAL": global_}), dentro]).rename("corr(sepal_length, sepal_width)").to_frame()

> **Puente P&E — paradoja de Simpson:** globalmente la correlación es **negativa**, pero dentro de
> cada especie es **positiva**. Mezclar grupos con distinta distribución puede invertir una relación.
> Por eso el EDA se mira **condicionado a Y**.

## 4. Feature Engineering

| Hallazgo del EDA | Feature | Justificación |
|---|---|---|
| Largo y ancho del pétalo separan (F muy alta) | `petal_area = petal_length × petal_width` | combina dos señales fuertes en una (tamaño) |
| La forma podría diferir entre especies | `petal_ratio = petal_length / petal_width` | captura forma, independiente de escala |
| Sépalo con correlación tipo Simpson | `sepal_ratio = sepal_length / sepal_width` | resume la forma del sépalo |

Son transformaciones **fila a fila** (no aprenden nada de los datos) ⇒ se pueden calcular antes del split sin leakage.

In [ ]:
df_fe = df.copy()
df_fe["petal_area"] = df_fe["petal_length"] * df_fe["petal_width"]
df_fe["petal_ratio"] = df_fe["petal_length"] / df_fe["petal_width"]
df_fe["sepal_ratio"] = df_fe["sepal_length"] / df_fe["sepal_width"]

nuevas = ["petal_area", "petal_ratio", "sepal_ratio"]
filas = []
for c in nuevas:
    F, p = stats.f_oneway(*[g[c] for _, g in df_fe.groupby("especie")])
    filas.append({"feature": c, "F": F, "asimetria": df_fe[c].skew()})
pd.DataFrame(filas).set_index("feature")

> **Atención:** `petal_ratio` queda **muy asimétrica** (asimetría > 2): en setosa el ancho del pétalo es
> chico, y dividir por valores chicos genera una cola larga. Para un modelo lineal podría convenir log;
> para árboles no importa. Se deja como decisión del Colab de cada modelo.

## 5. Leakage al cambiar de tarea

Si la tarea es **predecir `petal_width`** (regresión), `petal_area` y `petal_ratio` **contienen el target**:

In [ ]:
corr_leak = df_fe[["petal_area", "petal_ratio", "petal_length"]].corrwith(df_fe["petal_width"])
corr_leak.rename("corr con petal_width (target de regresión)").to_frame()

> **Puente P&E:** una feature que es **función del target** hace que P(Y | X) sea casi determinística
> en train, pero en producción esa feature **no existiría** sin conocer Y. Por eso `cargar("regresion")`
> las elimina y usa `especie` como feature categórica.

## 6. Preprocessing

- **Clasificación:** 7 numéricas → `StandardScaler`.
- **Regresión:** numéricas → `StandardScaler`; `especie` → `OneHotEncoder(drop="first")`
  (la especie eliminada es la referencia: cada coeficiente es una **diferencia de medias**).

**Regla de oro:** el preprocesador se **ajusta solo en train** (media y desvío son estimadores).

In [ ]:
X = df_fe.drop(columns=["especie"])
y = df_fe["especie"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

prep = ColumnTransformer([("num", StandardScaler(), list(X.columns))],
                         verbose_feature_names_out=False)
prep.fit(X_train)                     # <- solo train

Xt_train = pd.DataFrame(prep.transform(X_train), columns=prep.get_feature_names_out())
Xt_test = pd.DataFrame(prep.transform(X_test), columns=prep.get_feature_names_out())

pd.DataFrame({"media_train": Xt_train.mean(), "std_train": Xt_train.std(ddof=0),
              "media_test": Xt_test.mean(), "std_test": Xt_test.std(ddof=0)})

> **Puente P&E:** en train la media da 0 y el desvío 1 **por construcción**. En test **no**:
> son estimadores calculados en otra muestra. Si en test también dieran exactamente 0 y 1,
> sospechá que el scaler vio el test (leakage).

## 7. Verificación: el preprocesador funciona dentro de un Pipeline

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
modelos = {
    "Dummy (prior)": DummyClassifier(strategy="most_frequent"),
    "Logística": LogisticRegression(max_iter=1000),
}
res = {}
for nombre, m in modelos.items():
    pipe = Pipeline([("prep", prep), ("modelo", m)])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    res[nombre] = {"accuracy_media": s.mean(), "desvio": s.std()}
pd.DataFrame(res).T

## 8. Exportar el módulo `cap5_iris.py`

Consolida ETL + FE + preprocessing con la interfaz común.

In [ ]:
%%writefile cap5_iris.py
"""cap5_iris.py - ETL + FE + Preprocessing de IRIS (Capítulo 5, SIC).
Interfaz común: cargar(tarea), preprocesador(tarea, ...), dividir(...), INFO."""
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

SEED = 42
CLASES = ["setosa", "versicolor", "virginica"]

INFO = {
    "nombre": "iris",
    "tipo_caso": "clasificación multiclase limpia y balanceada (caso de laboratorio)",
    "tareas": {
        "clasificacion": "y = especie (0 setosa, 1 versicolor, 2 virginica)",
        "regresion": "y = petal_width (sin features derivadas del target)",
    },
    "metrica_sugerida": {"clasificacion": "accuracy", "regresion": "RMSE"},
}


def _etl():
    df = load_iris(as_frame=True).frame
    df.columns = ["sepal_length", "sepal_width", "petal_length",
                  "petal_width", "especie"]
    return df


def _feature_engineering(df):
    df = df.copy()
    df["petal_area"] = df["petal_length"] * df["petal_width"]
    df["petal_ratio"] = df["petal_length"] / df["petal_width"]
    df["sepal_ratio"] = df["sepal_length"] / df["sepal_width"]
    return df


def cargar(tarea="clasificacion"):
    df = _feature_engineering(_etl())
    if tarea == "clasificacion":
        y = df["especie"].astype(int)
        X = df.drop(columns=["especie"])
    elif tarea == "regresion":
        y = df["petal_width"]
        # petal_area y petal_ratio se calculan CON petal_width: serían leakage.
        X = df.drop(columns=["petal_width", "petal_area", "petal_ratio"])
        X["especie"] = X["especie"].map(dict(enumerate(CLASES)))
    else:
        raise ValueError("tarea debe ser 'clasificacion' o 'regresion'")
    return X, y


def preprocesador(tarea="clasificacion", escalar=True):
    X, _ = cargar(tarea)
    numericas = [c for c in X.columns if c != "especie"]
    pasos = [("num", StandardScaler() if escalar else "passthrough", numericas)]
    if "especie" in X.columns:
        pasos.append(("cat", OneHotEncoder(drop="first", handle_unknown="ignore",
                                           sparse_output=False), ["especie"]))
    return ColumnTransformer(pasos, verbose_feature_names_out=False)


def dividir(X, y, tarea="clasificacion", test_size=0.2):
    estratos = y if tarea == "clasificacion" else None
    return train_test_split(X, y, test_size=test_size,
                            random_state=SEED, stratify=estratos)

In [ ]:
import importlib, cap5_iris
importlib.reload(cap5_iris)
from cap5_iris import cargar, preprocesador, dividir, INFO

for tarea in ["clasificacion", "regresion"]:
    X_, y_ = cargar(tarea)
    Xtr, Xte, ytr, yte = dividir(X_, y_, tarea)
    p = preprocesador(tarea).fit(Xtr)
    print(f"{tarea:14s} X={X_.shape}  y={y_.name}  ->  features={len(p.get_feature_names_out())}")

# Consistencia con lo hecho paso a paso
X_mod, y_mod = cargar("clasificacion")
assert X_mod.equals(X) and y_mod.equals(y), "El módulo no coincide con el notebook"
print("OK: el módulo reproduce el notebook")
INFO

In [ ]:
# (Opcional) Guardar el módulo en Google Drive para reutilizarlo en los Colabs de modelos.
USAR_DRIVE = False

if USAR_DRIVE:
    from google.colab import drive
    import shutil, os
    drive.mount("/content/drive")
    destino = "/content/drive/MyDrive/SIC_cap5"
    os.makedirs(destino, exist_ok=True)
    shutil.copy("cap5_iris.py", destino)
    print("Módulo guardado en", destino)
    # En los Colabs de modelos:
    #   import sys; sys.path.append("/content/drive/MyDrive/SIC_cap5")
    #   from cap5_iris import cargar, preprocesador, dividir, INFO

## 9. Ejercicio propuesto

1. Agregá una feature `sepal_area` y medí su F en el ANOVA. ¿Aporta respecto de `petal_area`?
2. Repetí la sección 3.4 (Simpson) para `petal_length` vs. `sepal_width`.
3. Cambiá `test_size` a 0.5 y a 0.1: ¿cómo cambian la media y el desvío de la accuracy en la sección 7? Explicalo con el tamaño de muestra.
4. Agregá `preprocesador(..., escalar=False)` a un `DecisionTreeClassifier` y compará con el escalado. ¿Por qué da igual?

## 10. Cierre: tabla puente

| Concepto P&E | Herramienta ML en este Colab |
|---|---|
| Muestra de P(X, Y) | `load_iris`, ficha de calidad |
| Prior P(Y = k) | balance de clases, `DummyClassifier` |
| Momentos condicionales X \| Y | `groupby(...).agg(["mean","std"])` |
| ANOVA (entre vs. dentro) | ranking de features por F |
| Paradoja de Simpson | correlación global vs. por especie |
| Feature función del target | leakage en `cargar("regresion")` |
| Estimadores (media, desvío) | `StandardScaler` ajustado solo en train |

## 11. Autoevaluación
1. ¿Por qué el baseline de iris acierta 33%?
2. ¿Qué indica una F muy alta en el ANOVA?
3. ¿Por qué `petal_area` es válida para clasificar pero no para predecir `petal_width`?
4. ¿Por qué en test la media escalada no es exactamente 0?